## Loading raw Bronze data

Reads all JSON files from the Bronze volume into a single Spark DataFrame.
Each file represents one ingestion poll cycle and contains a nested `vessels`
array. `multiline=true` is required because each file is one JSON object
spanning multiple lines, not one JSON per line.

In [0]:
# Path to the raw Bronze JSON files, uploaded from local ingestion runs
bronze_path = "/Volumes/logistics_pipeline/bronze/vessel_positions_raw/"

df_bronze = spark.read.option("multiline", "true").json(bronze_path)

In [0]:
display(df_bronze)

In [0]:
# Exploding nested vessel arrays into individual rows
from pyspark.sql.functions import explode
df_exploded = df_bronze.select(explode("vessels").alias("vessel"))
display(df_exploded)

In [0]:
from pyspark.sql.functions import col

# Flattening fields and enforcing correct data types

df_flat = df_exploded.select(
    col("vessel.mmsi").cast("long").alias("mmsi"),
    col("vessel.imo").cast("long").alias("imo"),
    col("vessel.vessel_name").alias("vessel_name"),
    col("vessel.latitude").cast("double").alias("latitude"),
    col("vessel.longitude").cast("double").alias("longitude"),
    col("vessel.cog").cast("double").alias("cog"),
    col("vessel.sog").cast("double").alias("sog"),
    col("vessel.heading").cast("int").alias("heading"),
    col("vessel.nav_status").cast("int").alias("nav_status"),
    col("vessel.suspected_glitch").alias("suspected_glitch"),
    col("vessel.timestamp").cast("timestamp").alias("broadcast_timestamp"),
    col("vessel.processed_timestamp").cast("timestamp").alias("processed_timestamp"),
)

display(df_flat)

In [0]:
# Deduplication
from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("mmsi", "broadcast_timestamp").orderBy(desc("processed_timestamp"))

df_deduped = (
    df_flat
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1) # keeps only the latest processed version
    .drop("row_num") # remove other column
)

display(df_deduped)
print(f"Before dedup: {df_flat.count()} rows")
print(f"After dedup: {df_deduped.count()} rows")

## Translate nav_status codes into Business readable labels

`nav_status` is a numeric AIS navigational status code (0-15), standardized
under ITU-R M.1371. Mapped here to readable labels for downstream reporting and dashboards.

In [0]:
from pyspark.sql.functions import when

# Standard AIS navigational status codes (ITU-R M.1371)
df_labeled = df_deduped.withColumn(
    "nav_status_label",
    when(col("nav_status") == 0, "Under way using engine")
    .when(col("nav_status") == 1, "At anchor")
    .when(col("nav_status") == 2, "Not under command")
    .when(col("nav_status") == 3, "Restricted maneuverability")
    .when(col("nav_status") == 4, "Constrained by draught")
    .when(col("nav_status") == 5, "Moored")
    .when(col("nav_status") == 6, "Aground")
    .when(col("nav_status") == 7, "Engaged in fishing")
    .when(col("nav_status") == 8, "Under way sailing")
    .when(col("nav_status").isNull(), "Not reported")
    .otherwise("Other/reserved code")
)

display(df_labeled)

## Quarantining suspected data glitches

VesselAPI flags likely bad position reports via `suspected_glitch`. i want the rejected records to be visible, not silently
dropped, will split the data into two datasets: clean records for Gold layer use and
a quarantine table preserving flagged records for review/auditing.

In [0]:
df_silver_clean = df_labeled.filter(col("suspected_glitch") == False)
df_silver_quarantine = df_labeled.filter(col("suspected_glitch") == True)

print(f"Clean records: {df_silver_clean.count()}")
print(f"Quarantined (suspected glitch) records: {df_silver_quarantine.count()}")

display(df_silver_quarantine)

## Null handling

Explicit, documented decision for every column, rather than leaving null
handling implicit.

In [0]:
from pyspark.sql.functions import count, when as spark_when, isnan, col, sum as spark_sum

# Total row count in the clean Silver dataset our denominator for
# calculating null percentages per column
total_rows = df_silver_clean.count()

# For every column in the DataFrame, count how many rows have a null value
# in that specific column. spark_when/otherwise converts each null into a
# 1 (and non-null into 0), then spark_sum adds those up per column.
null_counts = df_silver_clean.select([
    spark_sum(spark_when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_silver_clean.columns
])

display(null_counts)
print(f"Total rows: {total_rows}")

## Null handling decisions (based on real data available at the time
## of this project, out of 1,067 total rows)

| Column | Null rate | Decision | Reasoning |
|---|---|---|---|
| mmsi, vessel_name, latitude, longitude, sog | 0% | No action needed | Always present in this dataset |
| suspected_glitch, broadcast_timestamp, processed_timestamp | 0% | No action needed | Always present |
| nav_status_label | 0% | No action needed | Nulls already converted to "Not reported" in Step 5 |
| **imo** | 59% | **Leave as null** | IMO numbers are only assigned to larger seagoing vessels under international convention. High null rate reflects that most tracked vessels are smaller/local craft (tugs, service vessels). This is expected, not a data quality issue. |
| **cog** | 36% | **Leave as null** | Course over ground is only meaningful for a moving vessel. Nulls are expected for stationary vessels. |
| **heading, nav_status** | 57% (identical count) | **Leave as null; flagged as a known limitation** | Likely come from the same AIS message type, vessels report both or neither.|

**General approach:** nulls are preserved (not dropped or defaulted) throughout Silver,
since forcing a fake value into a genuinely missing field would misrepresent the
data. Downstream consumers (Gold, dashboards) are expected to handle nulls
explicitly per their own use case, as I did for nav_status_label.

## Writing clean data to the Silver Delta table

In [0]:
silver_table = "logistics_pipeline.silver.vessel_positions_clean"

df_silver_clean.write.mode("overwrite").saveAsTable(silver_table)

print(f"Written to {silver_table}: {df_silver_clean.count()} rows")